In [ ]:
print('Bai 1')

TEXTS = [
    "hoc may va tri tue nhan tao",
    "tri tue nhan tao va khoa hoc du lieu",
    "lap trinh python cho may hoc",
    "phan tich du lieu bang python va numpy",
    "hoc sau hon ve ma tran va dai so tuyen tinh",
    "xay dung mo hinh hoc may tu du lieu",
    "xu ly ngon ngu tu nhien va van ban",
    "thuat toan tim kiem thong tin",
]

def tokenize(text: str) -> list[str]:
    return [word for word in text.lower().split() if word]

def build_bow_matrix(texts: list[str]) -> tuple[np.ndarray, list[str]]:
    vocab = sorted({word for text in texts for word in tokenize(text)})
    vocab_index = {word: idx for idx, word in enumerate(vocab)}
    matrix = np.zeros((len(texts), len(vocab)), dtype=float)
    for row, text in enumerate(texts):
        for word in tokenize(text):
            matrix[row, vocab_index[word]] += 1.0
    return matrix, vocab

def cosine_similarity_matrix(x: np.ndarray, y: np.ndarray | None = None) -> np.ndarray:
    if y is None:
        y = x
    x_norm = np.linalg.norm(x, axis=1, keepdims=True)
    y_norm = np.linalg.norm(y, axis=1, keepdims=True)
    x_norm[x_norm == 0] = 1.0
    y_norm[y_norm == 0] = 1.0
    return (x @ y.T) / (x_norm * y_norm.T)

def search_query(query: str, texts: list[str], top_k: int = 3) -> list[tuple[int, float, str]]:
    bow, vocab = build_bow_matrix(texts)
    vocab_index = {word: idx for idx, word in enumerate(vocab)}
    q_vec = np.zeros((1, len(vocab)), dtype=float)
    for word in tokenize(query):
        if word in vocab_index:
            q_vec[0, vocab_index[word]] += 1.0
    sims = cosine_similarity_matrix(q_vec, bow).ravel()
    order = np.argsort(-sims)
    return [(int(idx), float(sims[idx]), texts[idx]) for idx in order[:top_k]]

bow, vocab = build_bow_matrix(TEXTS)
print('Vocabulary size:', len(vocab))
print('Matrix shape:', bow.shape)
print("Top 3 for query 'hoc may python':")
for idx, score, text in search_query('hoc may python', TEXTS, top_k=3):
    print(f'  {idx + 1}: score={score:.4f} | {text}')


In [ ]:
print('Bai 2')

def reconstruct_image(image_matrix: np.ndarray, k: int) -> np.ndarray:
    u, s, vt = np.linalg.svd(image_matrix, full_matrices=False)
    return (u[:, :k] * s[:k]) @ vt[:k, :]

def lsa_coordinates(texts: list[str], n_components: int = 2) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    bow, _ = build_bow_matrix(texts)
    x_centered = bow - bow.mean(axis=0)
    u, s, vt = np.linalg.svd(x_centered, full_matrices=False)
    coords = u[:, :n_components] * s[:n_components]
    return x_centered, coords, vt

x = np.linspace(0, 1, 160)
y = np.linspace(0, 1, 120)
xv, yv = np.meshgrid(x, y)
synthetic_image = np.clip(0.65 * xv + 0.35 * np.sin(8 * math.pi * yv) * 0.2 + 0.15, 0, 1)
recon = reconstruct_image(synthetic_image, k=10)
print('Synthetic image shape:', synthetic_image.shape)
print('Reconstruction MSE:', float(np.mean((synthetic_image - recon) ** 2)))

x_centered, coords, vt = lsa_coordinates(TEXTS, n_components=2)
print('Centered matrix shape:', x_centered.shape)
print('Coords shape:', coords.shape)
print('First two singular vectors shape:', vt[:2].shape)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(synthetic_image, cmap='gray')
axes[0].set_title('Original image')
axes[0].axis('off')
axes[1].imshow(recon, cmap='gray')
axes[1].set_title('Reconstructed (k=10)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))
plt.scatter(coords[:, 0], coords[:, 1], s=80)
for idx in range(len(TEXTS)):
    plt.annotate(str(idx + 1), (coords[idx, 0], coords[idx, 1]), xytext=(4, 4), textcoords='offset points')
plt.axhline(0, color='gray', linewidth=0.8)
plt.axvline(0, color='gray', linewidth=0.8)
plt.title('2D LSA coordinates of texts')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.tight_layout()
plt.show()


In [ ]:
def reconstruct_image(image_matrix: np.ndarray, k: int) -> np.ndarray:
    u, s, vt = np.linalg.svd(image_matrix, full_matrices=False)
    return (u[:, :k] * s[:k]) @ vt[:k, :]

def lsa_coordinates(texts: list[str], n_components: int = 2) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    bow, _ = build_bow_matrix(texts)
    x_centered = bow - bow.mean(axis=0)
    u, s, vt = np.linalg.svd(x_centered, full_matrices=False)
    coords = u[:, :n_components] * s[:n_components]
    return x_centered, coords, vt

# Demo ảnh tự tạo để notebook chạy được ngay
x = np.linspace(0, 1, 160)
y = np.linspace(0, 1, 120)
xv, yv = np.meshgrid(x, y)
synthetic_image = np.clip(0.65 * xv + 0.35 * np.sin(8 * math.pi * yv) * 0.2 + 0.15, 0, 1)
recon = reconstruct_image(synthetic_image, k=10)
print('Synthetic image shape:', synthetic_image.shape)
print('Reconstruction MSE:', float(np.mean((synthetic_image - recon) ** 2)))

x_centered, coords, vt = lsa_coordinates(TEXTS, n_components=2)
print('Centered matrix shape:', x_centered.shape)
print('Coords shape:', coords.shape)
print('First two singular vectors shape:', vt[:2].shape)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(synthetic_image, cmap='gray')
axes[0].set_title('Original image')
axes[0].axis('off')
axes[1].imshow(recon, cmap='gray')
axes[1].set_title('Reconstructed (k=10)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 5))
plt.scatter(coords[:, 0], coords[:, 1], s=80)
for idx in range(len(TEXTS)):
    plt.annotate(str(idx + 1), (coords[idx, 0], coords[idx, 1]), xytext=(4, 4), textcoords='offset points')
plt.axhline(0, color='gray', linewidth=0.8)
plt.axvline(0, color='gray', linewidth=0.8)
plt.title('2D LSA coordinates of texts')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.tight_layout()
plt.show()